In [ ]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/ARR Data/Human Validation/humanData.xlsx" .

In [ ]:
import numpy as np
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score

In [ ]:
df = pd.read_excel("humanData.xlsx")

In [ ]:
annotator1_col = 'Hop_2_Target_Basis_of_Identity (A1)'
annotator2_col = 'Hop_2_Target_Basis_of_Identity (A2)'

In [ ]:
df = df[df["Hop_1_Triage (A1)"] != "بی‌ضرر"]
df = df[df["Hop_1_Triage (A2)"] != "بی‌ضرر"]
print(len(df))

68


In [ ]:
annotator1Identities = []
for annotator1Identity in df[annotator1_col]:
  annotator1Identities.extend(ast.literal_eval(annotator1Identity))
set(annotator1Identities)

{'other',
 'جنسیت',
 'حرفه/شغل',
 'دین/باور',
 'سوابق تحصیلی/دانشگاهی',
 'قومیت/ملیت',
 'ماهیت انسانی',
 'نقش اجتماعی/خانوادگی',
 'هویت فردی',
 'وابستگی سازمانی/نهادی',
 'وضعیت مهاجرت',
 'ویژگی ذهنی/رفتاری',
 'گرایش سیاسی'}

In [ ]:
annotator2Identities = []
for annotator2Identity in df[annotator2_col]:
  annotator2Identities.extend(ast.literal_eval(annotator2Identity))
set(annotator2Identities)

{'other',
 'حرفه/شغل',
 'دین/باور',
 'قومیت/ملیت',
 'نقش اجتماعی/خانوادگی',
 'هویت فردی',
 'وابستگی سازمانی/نهادی',
 'وضعیت مهاجرت',
 'گرایش سیاسی'}

In [ ]:
len(set(annotator1Identities))

13

In [ ]:
len(set(annotator2Identities))

9

In [ ]:
df[annotator1_col] = df[annotator1_col].apply(ast.literal_eval)
df[annotator2_col] = df[annotator2_col].apply(ast.literal_eval)

In [ ]:
# Remove "other" from both columns
df[annotator1_col] = df[annotator1_col].apply(
    lambda x: [label for label in x if label != 'other']
)

df[annotator2_col] = df[annotator2_col].apply(
    lambda x: [label for label in x if label != 'other']
)

# Remove rows where either side becomes empty
df_eval = df[
    df[annotator1_col].apply(len).gt(0) &
    df[annotator2_col].apply(len).gt(0)
].copy()

In [ ]:
mlb = MultiLabelBinarizer()

mlb.fit(list(df_eval[annotator1_col]) + list(df_eval[annotator2_col]))

y_annotator1 = mlb.transform(df_eval[annotator1_col])
y_annotator2 = mlb.transform(df_eval[annotator2_col])

In [ ]:
f1_micro = f1_score(y_annotator1, y_annotator2, average='micro')
f1_macro = f1_score(y_annotator1, y_annotator2, average='macro')

print(f"Micro F1: {f1_micro:.5f}")
print(f"Macro F1: {f1_macro:.5f}")

Micro F1: 0.66265
Macro F1: 0.51223
